In [2]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

REPO = r"C:\Users\FlorentinLAVAUD\Desktop\uniic\observatoire-competences-radar"

import pandas as pd 
pd.set_option('display.max_columns', None)

In [3]:
import duckdb
import pandas as pd
from pathlib import Path

Path(rf'{REPO}\data')

DATA_DIR        = Path(rf'{REPO}\data')
DB_PATH         = Path(rf'{REPO}\data\radar.duckdb') 
PARQUET_PATH    = Path(rf'{REPO}\data\lindustrie_offres.parquet')

print("DuckDB exists:  ", DB_PATH.exists())
print("Parquet exists: ", PARQUET_PATH.exists())

conn = duckdb.connect(str(DB_PATH))
print("Connexion DuckDB établie:", conn)

# Tables disponibles
print(conn.execute("SHOW TABLES").fetchall())

# Aperçu de la table scraper
df = conn.execute("SELECT * FROM lindustrie_offres LIMIT 5").df()
df

DuckDB exists:   True
Parquet exists:  True
Connexion DuckDB établie: <_duckdb.DuckDBPyConnection object at 0x00000246C1DB9130>
[('lindustrie_offres',), ('offres',)]


,id,source,titre,description,type_contrat,type_contrat_libelle,nature_contrat,experience_exige,experience_libelle,qualification_libelle,code_departement,lieu_travail_libelle,region,nom_acheteur,secteur,code_naf,code_rome,rome_libelle,appellation_libelle,nombre_postes,alternance,accessible_th,salaire_libelle,date_publication,date_creation,date_modification,reference_interne,raw_data,inserted_at
0,818038,lindustrie_recrute,"Chargé-e risques industriels, environnement di...",À propos de l’entreprise Cette offre d'emploi ...,CDI,CDI,None,None,2 à 5 ans,Master,91,Evry (91000),Île-de-France,Safran,None,None,None,None,None,<NA>,False,False,20 à 25k€ bruts /an,2026-06-10,2026-06-10,2026-06-10,206458-180832_1781111690,"{""@context"":""http://schema.org/"",""@type"":""JobP...",2026-06-29 10:35:36.404197
1,818052,lindustrie_recrute,Stage RH 6 mois - Talent Acquisition Specialis...,À propos de l’entreprise Cette offre d'emploi ...,STG,Stage,None,None,0 à 2 ans,Master,92,Malakoff (92240),Île-de-France,Safran,None,None,None,None,None,<NA>,False,False,25 à 30k€ bruts /an,2026-06-09,2026-06-09,2026-06-09,2026-170420_1780994592,"{""@context"":""http://schema.org/"",""@type"":""JobP...",2026-06-29 10:35:36.404197
2,818081,lindustrie_recrute,Technicien Bureau d'Etude en électricité indus...,"À propos de l’entreprise SOTEB, filiale du Gro...",CDI,CDI,Temps complet,None,2 à 5 ans,"BTS, DUT, BUT (Exigé)",71,Le-Creusot (71200),None,SOTEB,None,None,None,None,None,<NA>,False,False,None,NaT,NaT,NaT,so71techtrim2,"{""@context"":null,""@type"":null,""title"":null,""de...",2026-06-29 10:35:36.404197
3,818089,lindustrie_recrute,Chef de chantier électricité industrielle H/F,À propos de l’entreprise AMCR est une entrepri...,CDI,CDI,Temps complet,None,2 à 5 ans,Bac,34,Baillargues (34670),None,A.M.C.R./ACTEMIUM,None,None,None,None,None,<NA>,False,False,30 à 35k€ bruts /an,NaT,NaT,NaT,None,"{""@context"":null,""@type"":null,""title"":null,""de...",2026-06-29 10:35:36.404197
4,818096,lindustrie_recrute,Automaticien industriel H/F,À propos de l’entreprise AMCR est une entrepri...,CDI,CDI,Temps complet,None,2 à 5 ans,"BTS, DUT, BUT (Souhaité)",34,Baillargues (34670),None,A.M.C.R./ACTEMIUM,None,None,None,None,None,<NA>,False,False,35 à 45k€ bruts /an,NaT,NaT,NaT,None,"{""@context"":null,""@type"":null,""title"":null,""de...",2026-06-29 10:35:36.404197


In [4]:
import plotly.express as px

# Création du graphique en anneau
fig_contrat = px.pie(
    df,
    names="type_contrat_libelle",
    title="Répartition des offres par type de contrat",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel,
)

fig_contrat.update_traces(textinfo="percent+label")
fig_contrat.show()

In [5]:
# On filtre les valeurs manquantes pour plus de clarté
df_filtered = df.dropna(subset=["experience_libelle", "qualification_libelle"])

fig_xp_qualif = px.bar(
    df_filtered,
    x="qualification_libelle",
    color="experience_libelle",
    title="Niveau d'expérience requis selon la qualification",
    barmode="group",
    labels={
        "qualification_libelle": "Diplôme / Qualification",
        "experience_libelle": "Expérience",
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
)

fig_xp_qualif.update_layout(xaxis_tickangle=-45)
fig_xp_qualif.show()

In [6]:
import pandas as pd

# Conversion et agrégation par date (en ignorant les NaT)
df["date_publication_clean"] = pd.to_datetime(df["date_publication"], errors="coerce")
df_timeline = (
    df.dropna(subset=["date_publication_clean"])
    .groupby("date_publication_clean")
    .size()
    .reset_index(name="nb_offres")
)

fig_timeline = px.line(
    df_timeline,
    x="date_publication_clean",
    y="nb_offres",
    title="Évolution du nombre d'offres publiées",
    markers=True,
    labels={"date_publication_clean": "Date", "nb_offres": "Nombre d'offres"},
)

fig_timeline.show()

In [7]:
import re


# Fonction rapide pour extraire le premier chiffre du salaire (ex: 20 dans "20 à 25k€")
def extract_min_salary(text):
    if pd.isna(text) or text == "None":
        return None
    match = re.search(r"\d+", text)
    return int(match.group()) if match else None


df["salaire_min_k€"] = df["salaire_libelle"].apply(extract_min_salary)
df_salary = df.dropna(subset=["salaire_min_k€"])

# Histogramme des salaires minimums proposés
fig_salary = px.histogram(
    df_salary,
    x="salaire_min_k€",
    color="type_contrat_libelle",
    title="Distribution des salaires minimums (en k€) par type de contrat",
    labels={"salaire_min_k€": "Salaire minimum (k€ / an)"},
    nbins=10,
    barmode="overlay",
)

fig_salary.show()